**Lab type:** debug  
**Course:** DS104 — Statistics for Data Science  
**Lesson:** Sampling and the Central Limit Theorem  
**Task:** The sampling analysis pipeline below contains 3 bugs. Each runs without errors but introduces sampling bias, incorrect standard error calculations, or misleading CLT demonstrations. Find each bug, explain what it does wrong, and fix it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Population: 50,000 users across three regions
# UK has higher engagement scores due to product market fit
n_total = 50000
regions = np.random.choice(['US', 'UK', 'EU'], n_total, p=[0.60, 0.25, 0.15])

# Engagement scores: UK users score ~20 points higher on average
base_score = {'US': 55, 'UK': 75, 'EU': 60}
engagement = np.array([base_score[r] + np.random.normal(0, 15) for r in regions])
engagement = np.clip(engagement, 0, 100)

population = pd.DataFrame({'region': regions, 'engagement': engagement})

print(f'Population: {len(population)} users')
print(f'True mean engagement: {population.engagement.mean():.2f}')
print(f'By region:')
print(population.groupby('region').engagement.mean().round(2))

## Bug 1: Convenience sampling — filtering before sampling

An analyst wants to estimate mean engagement across all users, but first filters to users who have logged in this week (a convenience sample).

In [ ]:
# --- BUGGY CODE ---
# Simulate 'active this week' flag: UK users are 2× more likely to be active
active_prob = np.where(population.region == 'UK', 0.60, 0.30)
population['active_this_week'] = np.random.binomial(1, active_prob)

# BUG: filter to active users BEFORE sampling
active_users = population[population.active_this_week == 1]
sample_biased = active_users.sample(n=500, random_state=42)

print(f'True population mean: {population.engagement.mean():.2f}')
print(f'Biased sample mean:   {sample_biased.engagement.mean():.2f}')
print(f'Sample region mix: {sample_biased.region.value_counts(normalize=True).round(3).to_dict()}')
print(f'Population region mix: {population.region.value_counts(normalize=True).round(3).to_dict()}')

**Explanation:** Write your answer here — why does sampling from active users bias the estimate? Which region is over-represented, and how does that inflate the sample mean?

*(Write your answer here.)*

**Fix the bug:**

In [ ]:
# Correct: sample from the FULL population
sample_unbiased = population.sample(n=500, random_state=42)

print(f'True population mean: {population.engagement.mean():.2f}')
print(f'Unbiased sample mean: {sample_unbiased.engagement.mean():.2f}')
print(f'Sample region mix: {sample_unbiased.region.value_counts(normalize=True).round(3).to_dict()}')

## Bug 2: Standard error calculated as std instead of std/√n

The code below tries to compute the standard error of the mean but uses the wrong formula.

In [ ]:
# --- BUGGY CODE ---
sample = population.sample(n=100, random_state=1)

sample_mean = sample.engagement.mean()
sample_std = sample.engagement.std()

# BUG: SE is std, not std/sqrt(n)
se_buggy = sample_std

print(f'Sample mean: {sample_mean:.2f}')
print(f'Sample std: {sample_std:.2f}')
print(f'Reported SE (WRONG): {se_buggy:.2f}')
print(f'Correct SE (std/√n): {sample_std / np.sqrt(len(sample)):.2f}')
print(f'This overstates uncertainty by a factor of {se_buggy / (sample_std / np.sqrt(len(sample))):.0f}x')

**Explanation:** Write your answer here — what is the correct formula for standard error? By what factor is the buggy calculation wrong when n=100?

*(Write your answer here.)*

**Fix the bug:**

In [ ]:
# Correct SE formula: std / sqrt(n)
n = len(sample)
se_correct = sample_std / np.sqrt(n)

# Or use scipy directly
se_scipy = stats.sem(sample.engagement)

print(f'SE (manual): {se_correct:.2f}')
print(f'SE (scipy.stats.sem): {se_scipy:.2f}')

# Quick 95% CI using the correct SE
ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=sample_mean, scale=se_correct)
print(f'95% CI: [{ci_low:.2f}, {ci_high:.2f}]')

## Bug 3: CLT demonstration uses the population, not sample means

The code below claims to demonstrate the Central Limit Theorem but plots individual values rather than the distribution of sample means.

In [ ]:
# --- BUGGY CODE ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: population distribution
axes[0].hist(population.engagement, bins=60, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].set_title('Population distribution')

# BUG: this plots individual values from 1000 draws, not sample means
# It's just a sub-sample of the population, not a sampling distribution
individual_draws = [np.random.choice(population.engagement, 1)[0] for _ in range(1000)]
axes[1].hist(individual_draws, bins=40, color='coral', edgecolor='white', alpha=0.7)
axes[1].set_title('1000 individual draws (NOT sample means — this is wrong)')

plt.tight_layout()
plt.show()

print(f'Individual draws std: {np.std(individual_draws):.2f}')
print(f'This is the population std, not the SE of sample means.')

**Explanation:** Write your answer here — what should a CLT demonstration actually plot? What is the difference between a distribution of individual values and a sampling distribution of the mean?

*(Write your answer here.)*

**Fix the bug:**

In [ ]:
# Correct CLT demonstration: compute 1000 sample MEANS
sample_means_n30 = [population.engagement.sample(30).mean() for _ in range(1000)]
sample_means_n100 = [population.engagement.sample(100).mean() for _ in range(1000)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(population.engagement, bins=60, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].set_title(f'Population (std={population.engagement.std():.1f})')

axes[1].hist(sample_means_n30, bins=40, color='coral', edgecolor='white', alpha=0.7)
axes[1].set_title(f'Sample means n=30 (SE≈{np.std(sample_means_n30):.2f})')

axes[2].hist(sample_means_n100, bins=40, color='mediumpurple', edgecolor='white', alpha=0.7)
axes[2].set_title(f'Sample means n=100 (SE≈{np.std(sample_means_n100):.2f})')

plt.suptitle('CLT: Sampling distributions of the mean')
plt.tight_layout()
plt.show()

theoretical_se_30 = population.engagement.std() / np.sqrt(30)
theoretical_se_100 = population.engagement.std() / np.sqrt(100)
print(f'Theoretical SE (n=30):  σ/√30 = {theoretical_se_30:.2f}')
print(f'Simulated SE (n=30):  {np.std(sample_means_n30):.2f}')
print(f'Theoretical SE (n=100): σ/√100 = {theoretical_se_100:.2f}')
print(f'Simulated SE (n=100): {np.std(sample_means_n100):.2f}')

## Summary

> **Final question:** In one sentence each, state the three lessons from this lab.

1. 
2. 
3. 